# Summarizing a research project

When a consortium reports on a funding period or prepares a follow-up proposal, it has to say what its publications have established, which results rest on assumptions that were never tested, and which proof obligations are still open. Collecting this by hand means rereading every paper. With one SciLogic encoding per paper, most of it becomes a query.

This example uses the complete publication list of the Collaborative Research Center **CRC 1625** on compositionally complex solid solution surfaces for electrocatalysis: 62 publications from 2024 to 2026, covering combinatorial thin-film synthesis, electrochemistry, atomistic simulation and research data management. An LLM (Claude) encoded each publication from its PDF, following the annotation guide of the repository; figures were judged from their captions and the surrounding text. The encodings are in `crc1625/yaml/`, so nothing here calls an LLM.

The first two examples look at one paper and at a field. This one starts from a body of work.

In [ ]:
import re
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
import yaml

sys.path.insert(0, "../scripts")
import render

pd.set_option("display.max_colwidth", None)

encodings = {}
for path in sorted(Path("../crc1625/yaml").glob("*.yaml")):
    with open(path) as f:
        encodings[path.stem] = yaml.safe_load(f)

statements = [(key, s) for key, c in encodings.items() for s in c["statements"]]
relations = [r for c in encodings.values() for r in c["relations"]]
assessed = lambda s: (s.get("assessment") or {})

print(f"{len(encodings)} publications, {Counter(c['publication']['year'] for c in encodings.values())}")

Every encoding passes the referential checks of the renderer: all relations and proof obligations point to statements that exist, and every derived check evaluates. (Schema validation with `claimchain` needs the LinkML validator, which the book does not install; all 62 encodings pass it as well.)

In [ ]:
assert all(render.check(c) for c in encodings.values())

pd.DataFrame([
    {"encoding": key, "year": c["publication"]["year"], "title": c["publication"]["title"],
     "statements": len(c["statements"]), "relations": len(c["relations"]),
     "logic score": render.logic_score(c)[0], "overall": c["paper_assessment"]["overall_validity"]}
    for key, c in encodings.items()
]).style.hide(axis="index").format(precision=2)

## The project at a glance

The same counts that describe one paper in the [peer-review example](01-peer-review.ipynb) now describe the whole project.

In [ ]:
asserted = Counter(r["asserted_by"] for r in relations)
checks = [check for c in encodings.values() for check in render.run_checks(c)]
obligations = Counter(po["obligation_status"] for _, s in statements
                      for po in assessed(s).get("proof_obligations", []))
concerns = Counter(c["concern_type"] for _, s in statements for c in s.get("concerns", []))

print(f"Statements: {len(statements)}")
print(f"Relations:  {len(relations)}: {asserted['AUTHORS']} by the authors, "
      f"{asserted['ANNOTATOR'] + asserted['ANNOTATOR_BACKGROUND']} by the annotator {dict(asserted)}")
print(f"Checks:     {len(checks)} derived checks, {dict(Counter(c[2] for c in checks))}")
print(f"Proof obligations: {sum(obligations.values())} {dict(obligations)}")
print(f"Most frequent concerns: {dict(concerns.most_common(4))}")

### Verdicts on the main conclusions

Each paper states its main conclusions more than once: as a hypothesis in the introduction, as claims in the results and discussion, and again in the conclusion section, where the encoding links them with `RESTATES`. We leave out the main conclusions of the conclusion section, which mostly restate earlier ones, and count the rest. 9 of them are assessed YES.

In [ ]:
main = pd.DataFrame([
    {"encoding": key, "id": render.short(s["id"]), "type": s["statement_type"], "part": s.get("paper_part"),
     "verdict": assessed(s)["validity"], "text": s["text"]}
    for key, s in statements if s.get("claim_role") == "MAIN_CONCLUSION"
])
main = main[main.part != "CONCLUSION"]

verdicts = main.verdict.value_counts().reindex(render.RANK[::-1], fill_value=0)
print(f"{len(main)} main conclusions: {verdicts.to_dict()}")
pd.crosstab([main.type, main.part], main.verdict)[render.RANK[::-1]]

The verdict asks only whether the paper's own data support the claim as worded. PARTIAL dominates because most main conclusions generalize beyond what was measured (a trend shown, a mechanism proposed), not because the results are wrong.

### Numbers that do not follow

All derived checks recompute from the recorded values, but recomputing is also how the annotator finds reported numbers that do *not* follow from the paper's own values. These are recorded as `ARITHMETIC` concerns:

In [ ]:
arithmetic = pd.DataFrame([
    {"encoding": key, "statement": render.short(s["id"]), "priority": c.get("priority"), "concern": c["description"]}
    for key, s in statements for c in s.get("concerns", []) if c["concern_type"] == "ARITHMETIC"
])
print(f"{len(arithmetic)} reported numbers do not follow from the paper's own values")
arithmetic.style.hide(axis="index")

For example, a stress-to-strain conversion that is off by a factor of four (Baha 2026), or a pulse energy given in mJ instead of µJ (Osterloh 2026). Most are slips that do not change a conclusion, but each is a concrete erratum candidate.

## Key results by research theme

To summarize what the project has established, we group the publications into six research themes and pick the key results of each, with the verdict of the corresponding main conclusion. The grouping and the choice of results are ours (they follow Table 6 of the SciLogic paper); the verdicts are read from the encodings.

In [ ]:
themes = {
    "Synthesis and atomic-scale characterization platforms": (
        ["Li2024Investigation", "Akbarnejad2026Thermally", "Li2026Hidden", "Kar2026Understanding"],
        ["akbarnejad2026:CLAIM2", "li2026:CLAIM2", "kar2026:CLAIM3"]),
    "High-throughput and autonomous discovery": (
        ["Burgel2026Exploration", "Gao2026Nonmetal", "Thelen2026Autonomous"],
        ["burgel2026:CLAIM4", "gao2026:CLAIM3", "thelen2026:CLAIM1"]),
    "Descriptors of the electrochemical interface": (
        ["Liasi2026Work", "Ostergaard2024Coverage", "Gotz2026InSitu", "PerezMendoza2026Evaluation"],
        ["liasi2026:CLAIM2", "ostergaard2024:HYP1", "perezmendoza2026:CLAIM1"]),
    "Surface composition and restructuring": (
        ["Zerdoumi2026Surface", "Luan2024Revealing"],
        ["zerdoumi2026:CLAIM1", "luan2024:CLAIM1"]),
    "Atomistic simulation methods": (
        ["Ikeda2026Atomistic", "Todorova2026First"],
        ["ikeda2026:CLAIM1", "todorova2026:CLAIM2"]),
    "Research data and data-driven methods": (
        ["Dudarev2026Information", "Mohamed2026Field", "Stricker2025Computationally",
         "Zhang2025Composition", "Zhang2026Word2Vec"],
        ["dudarev2026b:CLAIM1", "mohamed2026:CLAIM2", "stricker2026:CLAIM1",
         "zhang2026:CLAIM1", "zhang2025:CLAIM2"]),
}

by_id = {s["id"]: (key, s) for key, s in statements}
missing = [sid for _, ids in themes.values() for sid in ids if sid not in by_id]
assert not missing, f"statements not found in the encodings: {missing}"

rows = []
for theme, (papers, ids) in themes.items():
    for sid in ids:
        key, s = by_id[sid]
        rows.append({"theme": theme, "encoding": key, "verdict": assessed(s)["validity"], "key result": s["text"]})
pd.DataFrame(rows).set_index(["theme", "encoding"])

The main conclusions assessed YES are the results the presented evidence supports most firmly:

In [ ]:
main[main.verdict == "YES"][["encoding", "id", "text"]].style.hide(axis="index")

A YES means that the paper's own data support the claim as worded, not that the result is the most important one. Judging impact would need expert assessment or citation data.

## Connections no single paper states

Reading the encodings side by side shows links between papers. SciLogic does not yet represent relations between encodings of different papers, so these connections were found by reading the encodings, not computed; the cells below only pull out the statements behind them.

In [ ]:
def show(*sids):
    for sid in sids:
        key, s = by_id[sid]
        verdict = f", {assessed(s)['validity']}" if assessed(s) else ""
        print(f"{key} {render.short(sid)} [{s['statement_type']}{verdict}]\n    {s['text']}")

**A shared descriptor.** Three independent studies, a computational one (Østergaard 2024), a combined experimental and computational descriptor study (Liasi 2026) and an in-situ spectroscopy study (Götz 2026), converge on hydrogen coverage as the quantity that determines hydrogen evolution on these surfaces. Together they suggest the adsorption energy at operating coverage as a shared descriptor for the next funding period.

In [ ]:
show("ostergaard2024:HYP1", "liasi2026:CLAIM4", "gotz2026:CLAIM2")

**A methodological sequence.** The Ag-Au-Pd-Pt system is a common model system and links several publications into a sequence: the native oxide on Si microtips fails at 400 °C, a thermally grown SiO<sub>2</sub> barrier extends the platform to 600 °C, and this makes the hidden low-temperature transformation pathway accessible.

In [ ]:
show("li2024:CLAIM1", "akbarnejad2026:CLAIM2", "li2026:CLAIM2")

**Tensions.** The same comparison exposes results that do not fit together:

- The Pt-rich phase in Ag-Au-Pd-Pt after 600 °C is reported with about 22 at.% Pt in one paper and about 40 at.% Pt in another.
- The descriptor study does not model the surface segregation that a companion paper measures for the same alloy family.
- Two papers on the data management system report FAIR self-assessment scores of 77 % and 82 %, for different instances.

In [ ]:
show("akbarnejad2026:OBS3", "li2026:OBS1")
print()
show("liasi2026:LIM1", "zerdoumi2026:CLAIM1")
print()
show("mohamed2026:OBS1", "dudarev2026:OBS2")

## Recurring open obligations

Open proof obligations and missing evidence that come up in several papers point to work packages that would strengthen several papers at once. Two examples: normalizing activities to the electrochemically active surface area (ECSA) rather than the geometric area, and comparing data-driven selection with a random baseline.

In [ ]:
def open_requests(pattern):
    # open or partially discharged obligations, and the missing evidence of each paper assessment
    rows = []
    for key, s in statements:
        for po in assessed(s).get("proof_obligations", []):
            if po["obligation_status"] != "DISCHARGED" and re.search(pattern, po["text"], re.I):
                rows.append((key, render.short(s["id"]), "proof obligation", po["text"]))
    for key, c in encodings.items():
        for m in c["paper_assessment"].get("missing_evidence", []):
            if re.search(pattern, m, re.I):
                rows.append((key, "paper", "missing evidence", m))
    return pd.DataFrame(rows, columns=["encoding", "statement", "kind", "text"])

open_requests(r"ECSA|electrochemically active").style.hide(axis="index")

In [ ]:
baseline = open_requests(r"random[- ](subset|baseline|initiali|and)|baseline")
print(f"{baseline.encoding.nunique()} papers with an open request for a random or simple baseline")
baseline.style.hide(axis="index")

## Try it on your own project

1. Encode the publications of your project: `claimchain paper.pdf -o encodings/paper.yaml` for each.
2. Point the loading cell above at your folder and rerun the notebook. The counts, verdicts, arithmetic concerns and recurring obligations follow without further work.
3. Group the publications into your own research themes, pick the key results, and read the encodings side by side for connections and tensions.

**Caveats.** The encodings are LLM drafts that neither the authors nor domain experts have reviewed, and their verdicts judge only whether each paper's own evidence supports its claims. The themes and the cross-paper connections were chosen by reading the encodings, not computed. Turning such a summary into a report or a proposal remains the consortium's task; the encodings make the starting point explicit and traceable to individual statements.